# CiteAgent VN: fine-tune reranker trên Kaggle

Tinh chỉnh `BAAI/bge-reranker-v2-m3` bằng câu hỏi đời thường do model cục bộ sinh ra, rồi so với reranker gốc
trên test v2 và held-out v3 (đo một lần mỗi cấu hình). **Không dùng API key nào** trừ khi chọn `GEN_BACKEND=openai`.

Trước khi chạy:
1. *Settings → Accelerator*: **GPU T4 x2** (hoặc P100/L4). *Settings → Internet*: **On**.
2. (Khuyến nghị) *Add Input*: dataset riêng tư chứa `citeagent_sources.zip`
   (`python scripts/pack_sources.py --with-embeddings` trên máy của bạn).
3. (Tùy chọn) *Add Input*: `queries.jsonl` đã sinh sẵn trên laptop để bỏ qua bước sinh câu hỏi.
4. Sửa ô cấu hình bên dưới nếu muốn: `GEN_MODEL` = `qwen3:8b` (mặc định, ~1–1,5 giờ sinh dữ liệu) hoặc
   `qwen3:14b` (tốt hơn, chậm gấp đôi).

Thời gian ước tính: cài đặt + build 20 phút, đo reranker gốc 15 phút, sinh câu hỏi 1–2 giờ, đào negative
30–60 phút, huấn luyện ~1 giờ, đo lại 15 phút.

Kết quả (mục *Output*): `citeagent_reranker_results.zip` (báo cáo), `citeagent_training_data.zip`
(câu hỏi và nhóm huấn luyện), `citeagent_reranker_ft.zip` (model ~1,1 GB).

In [ ]:
!nvidia-smi
!python --version

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/ThanhDatVN/chatbot_legal.git
%cd /kaggle/working/chatbot_legal
!git log --oneline -1

In [ ]:
import glob, os
os.environ['GEN_BACKEND'] = 'ollama'      # 'openai' cần OPENAI_API_KEY (Kaggle Secrets), ~0,3 USD
os.environ['GEN_MODEL'] = 'qwen3:8b'      # hoặc 'qwen3:14b'; với openai: 'gpt-4o-mini'
os.environ['EPOCHS'] = '1'
os.environ['LR'] = '1e-5'
found = glob.glob('/kaggle/input/**/citeagent_sources.zip', recursive=True)
os.environ['SOURCES_ZIP'] = found[0] if found else ''
queries = glob.glob('/kaggle/input/**/queries.jsonl', recursive=True)
os.environ['QUERIES_JSONL'] = queries[0] if queries else ''
if os.environ['GEN_BACKEND'] == 'openai':
    from kaggle_secrets import UserSecretsClient
    os.environ['OPENAI_API_KEY'] = UserSecretsClient().get_secret('OPENAI_API_KEY')
for k in ('GEN_BACKEND', 'GEN_MODEL', 'EPOCHS', 'LR', 'SOURCES_ZIP', 'QUERIES_JSONL'):
    print(k, '=', os.environ[k] or '(none)')

In [ ]:
!bash kaggle/finetune_reranker.sh 2>&1 | tee /kaggle/working/finetune.log

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('reports/reranker_experiment.md', encoding='utf-8').read()))